In [2]:
# Importet
import numpy as np
import pandas as pd

# Utforskning av datasets #


In [3]:

# Hämta datan från npz filen - allow 
sequence = np.load('data/segment_sequences.npz')

# Se vad som finns i filen
print(sequence.files)


['X_seq', 'y_seq', 'groups', 'segment_id']


In [4]:
# Läs in csv-filen features för att utforska. 
df_features = pd.read_csv('data/segment_features.csv', sep=None, engine='python')

# Visa de första raderna så man får en överblick
df_features.head()
print()

# Vi kollar hur klassfördelningen ser ut. (mode)
# Är något transportmedel överrepresenterat?
print("Fördelning av transportmedelel")
print(df_features['mode'].value_counts())
print()

# I procent
print("Fördelning i %")
print(df_features['mode'].value_counts(normalize=True) * 100)
print()
print("Kort reflektion:")
print("walk är väldigt överrepresenterat med ca 48 %, car lägst med ca 16 %")
print("walk kommer ev väga för tungt i modellträningen.")


Fördelning av transportmedelel
mode
walk    5412
bus     2298
bike    1859
car     1757
Name: count, dtype: int64

Fördelning i %
mode
walk    47.783860
bus     20.289599
bike    16.413562
car     15.512979
Name: proportion, dtype: float64

Kort reflektion:
walk är väldigt överrepresenterat med ca 48 %, car lägst med ca 16 %
walk kommer ev väga för tungt i modellträningen.


In [5]:
# Läs in csv-filen metadata för att utforska. 
df_metadata = pd.read_csv('data/segment_metadata.csv', sep=None, engine='python')

# Visa de första raderna så man får en överblick
df_metadata.head()

,segment_id,user_id,file_name,mode
0,0,10,20080331160008.plt,car
1,1,10,20080331160008.plt,car
2,2,10,20080402060926.plt,walk
3,3,10,20080402060926.plt,car
4,4,10,20080402060926.plt,bus


### Kommentar ###

**.npz filen:**  Vet inte om vi behöver den? Vet inte riktigt hur man ska göra med den datan eller hur man använder den?

**feature filen:** Här har vi all data vi behöver. features, targets(mode) och ev all userId:s och segmentsID:s.

**metadata filen:** Här har vi samma data som i featuredatan förutom alla features. Vi behöver kanske inte denna fil heller?

**Klassfördelningen:** Är lite skev, walk väger nog lite för tungt i datasetet mot de andra transportmedlen. Vi behöver nog ta hänsyn till det när vi splittar datan sen. Får fundera på hur vi ska lösa det.



# Task 1: Feature engineering and Machine Learning Model #


### Korrrelationsanalys ###

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt

# Vi gör först en korrealtionsanalys av alla features för att se vilka som verkar vara relevanta, och
# om vi hittar redundans. Gör det enklare att välja/göra nya features.

# Va tar bort kolumneran segement, id och filnamn då dessa inte är features. 
df_features_test = df_features.drop(columns=['segment_id', 'user_id', 'file_name'])
# Sen behöver vi konvertera mode (target) till siffror.
df_features_test['mode_encoded'] = df_features_test['mode'].astype('category').cat.codes
# kollar vilka siffro alla targets fick
print("Targets i siffror:")
print(dict(enumerate(df_features_test['mode'].astype('category').cat.categories)))
# tar bort gamla mode kolumnen för att kunna räkna. Den är ju en string
df_features_corr = df_features_test.drop(columns='mode')
# Räkna ut korrealtion
corr_matrix = df_features_corr.corr()

# Ritar upp en heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Korrelationsmatris för transportfeatures')
plt.show()
print()
print("Kommentar: Vi ser tydligt att fart featuresena korrelerar högt, över 93 %. Så där har vi redundans och kan slå ihop eller tar bort nån feature")
print("mid_speed_ratio som ser ut som att det är någon medelelåg hastighetsfeature i filen har ett ganska starkt negativt" \
"samband med målvariabeln (mode_encoded) vilket gör att den tydligt signalerar skillnaden mellan långsamma och sanbbare rörelser. " \
"gång - bil tex ") 


NameError: name 'df_features' is not defined

### Feature engineering ###

In [ ]:
# Väljer ut de features vi vill 
# Skapar en ny datafram för nya/valda features
df_final = pd.DataFrame()
# target/målvariabel
df_final['mode'] = df_features['mode']

# Speed - Använder p90 för det filtrerar bort alla extremvärden. Hade också 95 % korrelation mot avg. så vi undviker redundans
df_final['Speed'] = df_features['p90_speed_mps']

# Mellanfart - Läggs till för att det hade högt negativt samband med målvariabel. Så är troligtvis bra att ha med.
df_final['Mellanfart'] = df_features['mid_speed_ratio']

# Acceleration väljer ut max bara för tror det ger mest. Bör ju kunna skilja på gång och bil tydligt t ex- 
df_final['Max accelaration'] = df_features['max_acc_mps2']

# Distance
df_final['Distance'] = df_features['distance_m']

# Turning angle
df_final['Turning angle'] = df_features['avg_turn_deg']

# Stop ratio
df_final['Stop ratio'] = df_features['stop_ratio']


### Splitting the data and feature important plot ###


In [ ]:
# Förbered för ML-modell
X = df_final.drop('mode', axis=1)
y = df_final['mode']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Träna Random Forest för att få Feature Importance med random_state för reproducerbarhet
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Generera Feature Importance Plot
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title('Feature Importance - Task 1')
plt.barh(range(len(indices)), importances[indices], color='skyblue', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relativ betydelse för modellen')
plt.tight_layout()
plt.show()

print("Modellens träffsäkerhet:", model.score(X_test, y_test))